# 5 — Evaluate Model / 模型评估

**Chapter 26 — File 5 of 7 / 第26章 — 第5个文件（共7个）**

---

## Summary / 总结

This script demonstrates **load doc into memory**.

本脚本演示 **load doc into memory**。

---
## Step 1 — Step 1

In [ ]:
from numpy import argmax
from pickle import load
from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from keras.models import load_model
from nltk.translate.bleu_score import corpus_bleu

---
## Step 2 — load doc into memory

In [ ]:
def load_doc(filename):

---
## Step 3 — open the file as read only

In [ ]:
file = open(filename, 'r')

---
## Step 4 — read all text

In [ ]:
text = file.read()

---
## Step 5 — close the file

In [ ]:
file.close()
	return text

---
## Step 6 — load a pre-defined list of photo identifiers

In [ ]:
def load_set(filename):
	doc = load_doc(filename)
	dataset = list()

---
## Step 7 — process line by line

In [ ]:
for line in doc.split('\n'):

---
## Step 8 — skip empty lines

In [ ]:
if len(line) < 1:
			continue

---
## Step 9 — get the image identifier

In [ ]:
identifier = line.split('.')[0]
		dataset.append(identifier)
	return set(dataset)

---
## Step 10 — load clean descriptions into memory

In [ ]:
def load_clean_descriptions(filename, dataset):

---
## Step 11 — load document

In [ ]:
doc = load_doc(filename)
	descriptions = dict()
	for line in doc.split('\n'):

---
## Step 12 — split line by white space

In [ ]:
tokens = line.split()

---
## Step 13 — split id from description

In [ ]:
image_id, image_desc = tokens[0], tokens[1:]

---
## Step 14 — skip images not in the set

In [ ]:
if image_id in dataset:

---
## Step 15 — create list

In [ ]:
if image_id not in descriptions:
				descriptions[image_id] = list()

---
## Step 16 — wrap description in tokens

In [ ]:
desc = 'startseq ' + ' '.join(image_desc) + ' endseq'

---
## Step 17 — store

In [ ]:
descriptions[image_id].append(desc)
	return descriptions

---
## Step 18 — load photo features

In [ ]:
def load_photo_features(filename, dataset):

---
## Step 19 — load all features

In [ ]:
all_features = load(open(filename, 'rb'))

---
## Step 20 — filter features

In [ ]:
features = {k: all_features[k] for k in dataset}
	return features

---
## Step 21 — covert a dictionary of clean descriptions to a list of descriptions

In [ ]:
def to_lines(descriptions):
	all_desc = list()
	for key in descriptions.keys():
		[all_desc.append(d) for d in descriptions[key]]
	return all_desc

---
## Step 22 — fit a tokenizer given caption descriptions

In [ ]:
def create_tokenizer(descriptions):
	lines = to_lines(descriptions)
	tokenizer = Tokenizer()
	tokenizer.fit_on_texts(lines)
	return tokenizer

---
## Step 23 — calculate the length of the description with the most words

In [ ]:
def max_length(descriptions):
	lines = to_lines(descriptions)
	return max(len(d.split()) for d in lines)

---
## Step 24 — map an integer to a word

In [ ]:
def word_for_id(integer, tokenizer):
	for word, index in tokenizer.word_index.items():
		if index == integer:
			return word
	return None

---
## Step 25 — generate a description for an image

In [ ]:
def generate_desc(model, tokenizer, photo, max_length):

---
## Step 26 — seed the generation process

In [ ]:
in_text = 'startseq'

---
## Step 27 — iterate over the whole length of the sequence

In [ ]:
for _ in range(max_length):

---
## Step 28 — integer encode input sequence

In [ ]:
sequence = tokenizer.texts_to_sequences([in_text])[0]

---
## Step 29 — pad input

In [ ]:
sequence = pad_sequences([sequence], maxlen=max_length)

---
## Step 30 — predict next word

In [ ]:
yhat = model.predict([photo,sequence], verbose=0)

---
## Step 31 — convert probability to integer

In [ ]:
yhat = argmax(yhat)

---
## Step 32 — map integer to word

In [ ]:
word = word_for_id(yhat, tokenizer)

---
## Step 33 — stop if we cannot map the word

In [ ]:
if word is None:
			break

---
## Step 34 — append as input for generating the next word

In [ ]:
in_text += ' ' + word

---
## Step 35 — stop if we predict the end of the sequence

In [ ]:
if word == 'endseq':
			break
	return in_text

---
## Step 36 — remove start/end sequence tokens from a summary

In [ ]:
def cleanup_summary(summary):

---
## Step 37 — remove start of sequence token

In [ ]:
index = summary.find('startseq ')
	if index > -1:
		summary = summary[len('startseq '):]

---
## Step 38 — remove end of sequence token

In [ ]:
index = summary.find(' endseq')
	if index > -1:
		summary = summary[:index]
	return summary

---
## Step 39 — evaluate the skill of the model

In [ ]:
def evaluate_model(model, descriptions, photos, tokenizer, max_length):
	actual, predicted = list(), list()

---
## Step 40 — step over the whole set

In [ ]:
for key, desc_list in descriptions.items():

---
## Step 41 — generate description

In [ ]:
yhat = generate_desc(model, tokenizer, photos[key], max_length)

---
## Step 42 — clean up prediction

In [ ]:
yhat = cleanup_summary(yhat)

---
## Step 43 — store actual and predicted

In [ ]:
references = [cleanup_summary(d).split() for d in desc_list]
		actual.append(references)
		predicted.append(yhat.split())

---
## Step 44 — calculate BLEU score

In [ ]:
print('BLEU-1: %f' % corpus_bleu(actual, predicted, weights=(1.0, 0, 0, 0)))
	print('BLEU-2: %f' % corpus_bleu(actual, predicted, weights=(0.5, 0.5, 0, 0)))
	print('BLEU-3: %f' % corpus_bleu(actual, predicted, weights=(0.3, 0.3, 0.3, 0)))
	print('BLEU-4: %f' % corpus_bleu(actual, predicted, weights=(0.25, 0.25, 0.25, 0.25)))

---
## Step 45 — load training dataset (6K)

In [ ]:
filename = 'Flickr8k_text/Flickr_8k.trainImages.txt'
train = load_set(filename)
print('Dataset: %d' % len(train))

---
## Step 46 — descriptions

In [ ]:
train_descriptions = load_clean_descriptions('descriptions.txt', train)
print('Descriptions: train=%d' % len(train_descriptions))

---
## Step 47 — prepare tokenizer

In [ ]:
tokenizer = create_tokenizer(train_descriptions)
vocab_size = len(tokenizer.word_index) + 1
print('Vocabulary Size: %d' % vocab_size)

---
## Step 48 — determine the maximum sequence length

In [ ]:
max_length = max_length(train_descriptions)
print('Description Length: %d' % max_length)

---
## Step 49 — load test set

In [ ]:
filename = 'Flickr8k_text/Flickr_8k.testImages.txt'
test = load_set(filename)
print('Dataset: %d' % len(test))

---
## Step 50 — descriptions

In [ ]:
test_descriptions = load_clean_descriptions('descriptions.txt', test)
print('Descriptions: test=%d' % len(test_descriptions))

---
## Step 51 — photo features

In [ ]:
test_features = load_photo_features('features.pkl', test)
print('Photos: test=%d' % len(test_features))

---
## Step 52 — load the model

In [ ]:
filename = 'model.h5'
model = load_model(filename)

---
## Step 53 — evaluate model

In [ ]:
evaluate_model(model, test_descriptions, test_features, tokenizer, max_length)

---
## Learning Notes / 学习笔记

- **概念**: load doc into memory 是机器学习中的常用技术。  
  *load doc into memory is a common technique in machine learning.*

- **ML 应用**: 本示例展示了如何在实践中应用该技术。  
  *This example shows how to apply the technique in practice.*

---
## Complete Code / 完整代码一览

Below is the full code for quick reference. / 以下是完整代码，供快速参考。

In [ ]:
# ===============================
# Evaluate Model / 模型评估
# Complete Code / 完整代码
# ===============================

from numpy import argmax
from pickle import load
from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from keras.models import load_model
from nltk.translate.bleu_score import corpus_bleu

# load doc into memory
def load_doc(filename):
	# open the file as read only
	file = open(filename, 'r')
	# read all text
	text = file.read()
	# close the file
	file.close()
	return text

# load a pre-defined list of photo identifiers
def load_set(filename):
	doc = load_doc(filename)
	dataset = list()
	# process line by line
	for line in doc.split('\n'):
		# skip empty lines
		if len(line) < 1:
			continue
		# get the image identifier
		identifier = line.split('.')[0]
		dataset.append(identifier)
	return set(dataset)

# load clean descriptions into memory
def load_clean_descriptions(filename, dataset):
	# load document
	doc = load_doc(filename)
	descriptions = dict()
	for line in doc.split('\n'):
		# split line by white space
		tokens = line.split()
		# split id from description
		image_id, image_desc = tokens[0], tokens[1:]
		# skip images not in the set
		if image_id in dataset:
			# create list
			if image_id not in descriptions:
				descriptions[image_id] = list()
			# wrap description in tokens
			desc = 'startseq ' + ' '.join(image_desc) + ' endseq'
			# store
			descriptions[image_id].append(desc)
	return descriptions

# load photo features
def load_photo_features(filename, dataset):
	# load all features
	all_features = load(open(filename, 'rb'))
	# filter features
	features = {k: all_features[k] for k in dataset}
	return features

# covert a dictionary of clean descriptions to a list of descriptions
def to_lines(descriptions):
	all_desc = list()
	for key in descriptions.keys():
		[all_desc.append(d) for d in descriptions[key]]
	return all_desc

# fit a tokenizer given caption descriptions
def create_tokenizer(descriptions):
	lines = to_lines(descriptions)
	tokenizer = Tokenizer()
	tokenizer.fit_on_texts(lines)
	return tokenizer

# calculate the length of the description with the most words
def max_length(descriptions):
	lines = to_lines(descriptions)
	return max(len(d.split()) for d in lines)

# map an integer to a word
def word_for_id(integer, tokenizer):
	for word, index in tokenizer.word_index.items():
		if index == integer:
			return word
	return None

# generate a description for an image
def generate_desc(model, tokenizer, photo, max_length):
	# seed the generation process
	in_text = 'startseq'
	# iterate over the whole length of the sequence
	for _ in range(max_length):
		# integer encode input sequence
		sequence = tokenizer.texts_to_sequences([in_text])[0]
		# pad input
		sequence = pad_sequences([sequence], maxlen=max_length)
		# predict next word
		yhat = model.predict([photo,sequence], verbose=0)
		# convert probability to integer
		yhat = argmax(yhat)
		# map integer to word
		word = word_for_id(yhat, tokenizer)
		# stop if we cannot map the word
		if word is None:
			break
		# append as input for generating the next word
		in_text += ' ' + word
		# stop if we predict the end of the sequence
		if word == 'endseq':
			break
	return in_text

# remove start/end sequence tokens from a summary
def cleanup_summary(summary):
	# remove start of sequence token
	index = summary.find('startseq ')
	if index > -1:
		summary = summary[len('startseq '):]
	# remove end of sequence token
	index = summary.find(' endseq')
	if index > -1:
		summary = summary[:index]
	return summary

# evaluate the skill of the model
def evaluate_model(model, descriptions, photos, tokenizer, max_length):
	actual, predicted = list(), list()
	# step over the whole set
	for key, desc_list in descriptions.items():
		# generate description
		yhat = generate_desc(model, tokenizer, photos[key], max_length)
		# clean up prediction
		yhat = cleanup_summary(yhat)
		# store actual and predicted
		references = [cleanup_summary(d).split() for d in desc_list]
		actual.append(references)
		predicted.append(yhat.split())
	# calculate BLEU score
	print('BLEU-1: %f' % corpus_bleu(actual, predicted, weights=(1.0, 0, 0, 0)))
	print('BLEU-2: %f' % corpus_bleu(actual, predicted, weights=(0.5, 0.5, 0, 0)))
	print('BLEU-3: %f' % corpus_bleu(actual, predicted, weights=(0.3, 0.3, 0.3, 0)))
	print('BLEU-4: %f' % corpus_bleu(actual, predicted, weights=(0.25, 0.25, 0.25, 0.25)))

# load training dataset (6K)
filename = 'Flickr8k_text/Flickr_8k.trainImages.txt'
train = load_set(filename)
print('Dataset: %d' % len(train))
# descriptions
train_descriptions = load_clean_descriptions('descriptions.txt', train)
print('Descriptions: train=%d' % len(train_descriptions))
# prepare tokenizer
tokenizer = create_tokenizer(train_descriptions)
vocab_size = len(tokenizer.word_index) + 1
print('Vocabulary Size: %d' % vocab_size)
# determine the maximum sequence length
max_length = max_length(train_descriptions)
print('Description Length: %d' % max_length)

# load test set
filename = 'Flickr8k_text/Flickr_8k.testImages.txt'
test = load_set(filename)
print('Dataset: %d' % len(test))
# descriptions
test_descriptions = load_clean_descriptions('descriptions.txt', test)
print('Descriptions: test=%d' % len(test_descriptions))
# photo features
test_features = load_photo_features('features.pkl', test)
print('Photos: test=%d' % len(test_features))

# load the model
filename = 'model.h5'
model = load_model(filename)
# evaluate model
evaluate_model(model, test_descriptions, test_features, tokenizer, max_length)

---

➡️ **Next / 下一步**: File 6 of 7